In [ ]:
from pathlib import Path
import json

import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt


# ============================================================
# 1. Utilidades básicas
# ============================================================

def ensure_uint8_rgb(img):
    """Asegura formato RGB uint8."""
    img = np.asarray(img)

    if img.ndim != 3 or img.shape[2] != 3:
        raise ValueError("La imagen debe tener forma H x W x 3")

    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    return img


def odd_kernel_size(k):
    """Convierte un tamaño de kernel en entero impar >= 1."""
    k = int(round(k))
    if k < 1:
        k = 1
    if k % 2 == 0:
        k += 1
    return k


def overlay_mask(img_rgb, mask, color=(255, 255, 0), alpha=0.45):
    """Superpone una máscara binaria sobre una imagen RGB."""
    img_rgb = ensure_uint8_rgb(img_rgb)
    mask = mask.astype(bool)

    out = img_rgb.copy().astype(np.float32)
    color = np.array(color, dtype=np.float32)

    out[mask] = (1 - alpha) * out[mask] + alpha * color
    return np.clip(out, 0, 255).astype(np.uint8)


def draw_contour_overlay(
    img_rgb,
    contour,
    color=(255, 255, 0),
    thickness=2,
    show_vertices=False,
    vertex_color=(255, 0, 0),
    vertex_radius=2,
):
    """Dibuja un contorno OpenCV sobre una imagen RGB."""
    img_rgb = ensure_uint8_rgb(img_rgb)
    out = img_rgb.copy()

    if contour is not None and len(contour) > 0:
        cv2.drawContours(out, [contour.astype(np.int32)], -1, color, thickness=thickness)

        if show_vertices:
            pts = contour.reshape(-1, 2)
            step = max(1, len(pts) // 400)
            for x, y in pts[::step]:
                cv2.circle(out, (int(x), int(y)), vertex_radius, vertex_color, -1)

    return out


def compute_border_median_color(img_rgb, border_width=10):
    """Calcula el color mediano del borde de la imagen."""
    img_rgb = ensure_uint8_rgb(img_rgb)
    h, w = img_rgb.shape[:2]
    bw = int(max(1, min(border_width, h // 2, w // 2)))

    border_pixels = np.concatenate([
        img_rgb[:bw, :, :].reshape(-1, 3),
        img_rgb[-bw:, :, :].reshape(-1, 3),
        img_rgb[:, :bw, :].reshape(-1, 3),
        img_rgb[:, -bw:, :].reshape(-1, 3),
    ], axis=0)

    return np.median(border_pixels.astype(np.float32), axis=0)


def detect_border_connected_black(
    img_rgb,
    black_v_threshold=35,
    black_mean_threshold=45,
    border_width=5,
):
    """
    Detecta padding negro conectado al borde para NO confundirlo con tejido.

    Esto elimina zonas negras del escáner o del padding exterior.
    No elimina tejido oscuro si no está conectado al borde.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    V = hsv[:, :, 2]
    mean_rgb = img_rgb.mean(axis=2)

    raw_black = (V <= black_v_threshold) & (mean_rgb <= black_mean_threshold)

    h, w = raw_black.shape
    bw = int(max(1, border_width))
    border_mask = np.zeros((h, w), dtype=bool)
    border_mask[:bw, :] = True
    border_mask[-bw:, :] = True
    border_mask[:, :bw] = True
    border_mask[:, -bw:] = True

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        raw_black.astype(np.uint8), connectivity=8
    )

    if num_labels <= 1:
        return np.zeros_like(raw_black, dtype=bool)

    border_labels = np.unique(labels[raw_black & border_mask])
    border_labels = border_labels[border_labels != 0]

    if len(border_labels) == 0:
        return np.zeros_like(raw_black, dtype=bool)

    return np.isin(labels, border_labels)


def fill_holes_binary(mask):
    """
    Rellena huecos internos de una máscara binaria.

    Para contorno externo del espécimen suele interesar True,
    porque queremos una silueta cerrada.
    """
    mask_u8 = (mask.astype(np.uint8) * 255)
    h, w = mask_u8.shape

    flood = mask_u8.copy()
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)

    # Si la esquina (0, 0) cae dentro del tejido, probamos otras esquinas.
    seeds = [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)]
    for seed in seeds:
        x, y = seed
        if flood[y, x] == 0:
            cv2.floodFill(flood, flood_mask, seed, 255)

    holes = cv2.bitwise_not(flood)
    filled = cv2.bitwise_or(mask_u8, holes)
    return filled > 0


def component_stats(mask):
    """Devuelve estadísticas de componentes conectadas, excluyendo fondo."""
    mask = mask.astype(bool)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )

    comps = []
    for lab in range(1, num_labels):
        comps.append({
            "label": int(lab),
            "area": int(stats[lab, cv2.CC_STAT_AREA]),
            "x": int(stats[lab, cv2.CC_STAT_LEFT]),
            "y": int(stats[lab, cv2.CC_STAT_TOP]),
            "w": int(stats[lab, cv2.CC_STAT_WIDTH]),
            "h": int(stats[lab, cv2.CC_STAT_HEIGHT]),
            "centroid_x": float(centroids[lab][0]),
            "centroid_y": float(centroids[lab][1]),
        })

    comps = sorted(comps, key=lambda c: c["area"], reverse=True)
    return labels, comps


def keep_largest_components(mask, keep_n=1, min_area_px=0, min_area_frac=0.0):
    """
    Mantiene las N componentes más grandes por encima de un umbral.

    Por defecto keep_n=1 porque normalmente hay un único espécimen principal.
    Si tu muestra tiene varios fragmentos separados, sube keep_n o pon keep_n=None.
    """
    mask = mask.astype(bool)
    h, w = mask.shape
    min_area = max(int(min_area_px), int(min_area_frac * h * w))

    labels, comps = component_stats(mask)

    if len(comps) == 0:
        return np.zeros_like(mask, dtype=bool), []

    selected = []
    for c in comps:
        if c["area"] >= min_area:
            selected.append(c)

    if keep_n is not None:
        selected = selected[:int(keep_n)]

    out = np.zeros_like(mask, dtype=bool)
    for c in selected:
        out |= labels == c["label"]

    return out, selected


# ============================================================
# 2. Lectura de WSI en un level adecuado
# ============================================================

def choose_level_by_max_side(slide, max_analysis_side=3500):
    """
    Elige el primer level cuyo lado mayor sea <= max_analysis_side.

    Si ninguno cumple, devuelve el level más bajo disponible.
    """
    dims = slide.level_dimensions

    for level, (w, h) in enumerate(dims):
        if max(w, h) <= max_analysis_side:
            return level

    return len(dims) - 1


def read_wsi_level(slide_path, target_level=None, max_analysis_side=3500):
    """
    Abre una WSI y lee completa una resolución baja/media.
    """
    slide_path = Path(slide_path)
    slide = openslide.OpenSlide(str(slide_path))

    if target_level is None:
        target_level = choose_level_by_max_side(slide, max_analysis_side=max_analysis_side)

    if target_level < 0 or target_level >= len(slide.level_dimensions):
        raise ValueError(f"target_level={target_level} no existe. Levels disponibles: {len(slide.level_dimensions)}")

    lvl_w, lvl_h = slide.level_dimensions[target_level]
    downsample = float(slide.level_downsamples[target_level])

    print("\n==============================")
    print("LECTURA WSI")
    print("==============================")
    print(f"Slide: {slide_path}")
    print(f"Level dimensions: {slide.level_dimensions}")
    print(f"Level downsamples: {slide.level_downsamples}")
    print(f"Level usado: {target_level}")
    print(f"Dimensiones level: {lvl_w} x {lvl_h}")
    print(f"Downsample respecto a level 0: {downsample:.4f}")

    img = slide.read_region((0, 0), target_level, (lvl_w, lvl_h)).convert("RGB")
    img_rgb = np.array(img)

    return slide, img_rgb, target_level, downsample


# ============================================================
# 3. Segmentación global tejido vs fondo para H&E
# ============================================================

def segment_he_tissue_global(
    img_rgb,

    # Fondo / color
    border_width_for_bg=10,
    dist_to_bg_threshold=30,

    # HSV
    sat_threshold=18,
    value_upper_threshold=245,

    # Optical density aproximada
    od_threshold=0.10,
    min_saturation_for_od=5,

    # Padding negro conectado a borde
    black_v_threshold=35,
    black_mean_threshold=45,
    black_border_width=5,

    # Morfología
    open_kernel_size=5,
    close_kernel_size=21,
    smooth_kernel_size=9,

    # Componentes
    keep_n_components=1,
    min_component_area_px=5000,
    min_component_area_frac=0.0005,

    fill_internal_holes=True,
):
    """
    Segmenta el espécimen completo en una imagen RGB de un level de WSI H&E.

    La máscara se basa en tres señales:
    1. Distancia RGB al color mediano del borde.
    2. Saturación HSV.
    3. Optical density aproximada.

    Además excluye padding negro conectado al borde.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)
    h, w = img_rgb.shape[:2]

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)

    bg_color = compute_border_median_color(img_rgb, border_width=border_width_for_bg)

    img_float = img_rgb.astype(np.float32)
    dist_to_bg = np.linalg.norm(img_float - bg_color[None, None, :], axis=2)

    # OD aproximada. El +1 evita log(0).
    img_norm = (img_float + 1.0) / 255.0
    od = -np.log(img_norm)
    od_mean = od.mean(axis=2)

    black_padding_mask = detect_border_connected_black(
        img_rgb,
        black_v_threshold=black_v_threshold,
        black_mean_threshold=black_mean_threshold,
        border_width=black_border_width,
    )

    # Tres criterios complementarios.
    mask_sat = (S >= sat_threshold) & (V <= value_upper_threshold)
    mask_dist = (dist_to_bg >= dist_to_bg_threshold) & (V <= 252)
    mask_od = (od_mean >= od_threshold) & (S >= min_saturation_for_od)

    raw_candidate = (mask_sat | mask_dist | mask_od)

    # Excluir padding negro de escáner conectado al borde.
    raw_candidate = raw_candidate & (~black_padding_mask)

    # Morfología inicial.
    open_k = odd_kernel_size(open_kernel_size)
    close_k = odd_kernel_size(close_kernel_size)
    smooth_k = odd_kernel_size(smooth_kernel_size)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (smooth_k, smooth_k))

    mask_morph = cv2.morphologyEx(raw_candidate.astype(np.uint8), cv2.MORPH_OPEN, kernel_open) > 0
    mask_morph = cv2.morphologyEx(mask_morph.astype(np.uint8), cv2.MORPH_CLOSE, kernel_close) > 0

    # Mantener componente(s) principal(es).
    mask_components, selected_components = keep_largest_components(
        mask_morph,
        keep_n=keep_n_components,
        min_area_px=min_component_area_px,
        min_area_frac=min_component_area_frac,
    )

    # Rellenar huecos para obtener silueta externa.
    if fill_internal_holes:
        mask_filled = fill_holes_binary(mask_components)
    else:
        mask_filled = mask_components.copy()

    # Suavizado final del borde.
    mask_final = cv2.morphologyEx(mask_filled.astype(np.uint8), cv2.MORPH_CLOSE, kernel_smooth) > 0
    mask_final = cv2.morphologyEx(mask_final.astype(np.uint8), cv2.MORPH_OPEN, kernel_smooth) > 0

    debug = {
        "bg_color": bg_color,
        "hsv_H": H,
        "hsv_S": S,
        "hsv_V": V,
        "dist_to_bg": dist_to_bg,
        "od_mean": od_mean,
        "mask_sat": mask_sat,
        "mask_dist": mask_dist,
        "mask_od": mask_od,
        "black_padding_mask": black_padding_mask,
        "raw_candidate_mask": raw_candidate,
        "mask_after_morphology": mask_morph,
        "mask_components": mask_components,
        "mask_final": mask_final,
        "selected_components": selected_components,
    }

    print("\n==============================")
    print("SEGMENTACIÓN GLOBAL H&E")
    print("==============================")
    print(f"Imagen level: {w} x {h}")
    print(f"Color mediano de borde RGB: {bg_color.round(2).tolist()}")
    print(f"Pixels black padding borde: {int(black_padding_mask.sum())}")
    print(f"Pixels mask_sat: {int(mask_sat.sum())}")
    print(f"Pixels mask_dist: {int(mask_dist.sum())}")
    print(f"Pixels mask_od: {int(mask_od.sum())}")
    print(f"Pixels raw_candidate tras quitar negro: {int(raw_candidate.sum())}")
    print(f"Pixels tras morfología: {int(mask_morph.sum())}")
    print(f"Pixels máscara final: {int(mask_final.sum())}")

    print("\nComponentes seleccionadas:")
    if len(selected_components) == 0:
        print("  Ninguna componente seleccionada.")
    else:
        for i, c in enumerate(selected_components, start=1):
            print(
                f"  #{i}: area={c['area']} | "
                f"bbox=(x={c['x']}, y={c['y']}, w={c['w']}, h={c['h']}) | "
                f"centroid=({c['centroid_x']:.1f}, {c['centroid_y']:.1f})"
            )

    return mask_final, debug


# ============================================================
# 4. Extracción de contorno global cerrado
# ============================================================

def extract_external_contour_from_mask(
    mask,
    approx_epsilon_frac=0.001,
    contour_line_thickness=2,
):
    """
    Extrae el contorno externo principal de una máscara binaria.

    approx_epsilon_frac controla la simplificación:
    - 0.0000: no simplificar, máximo detalle.
    - 0.0005-0.002: suele ir bien para registro por silueta.
    - >0.005: puede simplificar demasiado.
    """
    mask = mask.astype(bool)
    h, w = mask.shape

    contours, _ = cv2.findContours(
        mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )

    if len(contours) == 0:
        raise ValueError("No se encontró ningún contorno externo en la máscara final.")

    raw_contour = max(contours, key=cv2.contourArea)

    raw_area = float(cv2.contourArea(raw_contour))
    raw_perimeter = float(cv2.arcLength(raw_contour, closed=True))

    if approx_epsilon_frac is not None and approx_epsilon_frac > 0:
        epsilon = float(approx_epsilon_frac) * raw_perimeter
        contour = cv2.approxPolyDP(raw_contour, epsilon=epsilon, closed=True)
    else:
        epsilon = 0.0
        contour = raw_contour

    contour_area = float(cv2.contourArea(contour))
    contour_perimeter = float(cv2.arcLength(contour, closed=True))

    contour_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(contour_mask, [contour], -1, 255, thickness=int(contour_line_thickness))
    contour_mask = contour_mask > 0

    x, y, bw, bh = cv2.boundingRect(contour)
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
    else:
        cx, cy = np.nan, np.nan

    print("\n==============================")
    print("CONTORNO GLOBAL")
    print("==============================")
    print(f"Puntos contorno bruto: {len(raw_contour)}")
    print(f"Área contorno bruto: {raw_area:.2f} px²")
    print(f"Perímetro contorno bruto: {raw_perimeter:.2f} px")
    print(f"approx_epsilon_frac: {approx_epsilon_frac}")
    print(f"epsilon usado: {epsilon:.4f} px")
    print(f"Puntos contorno final: {len(contour)}")
    print(f"Área contorno final: {contour_area:.2f} px²")
    print(f"Perímetro contorno final: {contour_perimeter:.2f} px")
    print(f"BBox contorno: x={x}, y={y}, w={bw}, h={bh}")
    print(f"Centroide contorno: ({cx:.2f}, {cy:.2f})")

    info = {
        "raw_contour": raw_contour,
        "contour": contour,
        "contour_mask": contour_mask,
        "raw_area_px2": raw_area,
        "raw_perimeter_px": raw_perimeter,
        "area_px2": contour_area,
        "perimeter_px": contour_perimeter,
        "bbox": {"x": int(x), "y": int(y), "w": int(bw), "h": int(bh)},
        "centroid": {"x": float(cx), "y": float(cy)},
        "approx_epsilon_px": float(epsilon),
    }

    return info


def contour_to_xy(contour):
    """Convierte contorno OpenCV Nx1x2 a array Nx2."""
    return contour.reshape(-1, 2).astype(np.float32)


def scale_contour_to_level0(contour, downsample):
    """Escala puntos de contorno desde el level usado a coordenadas de level 0."""
    pts = contour_to_xy(contour)
    return pts * float(downsample)


# ============================================================
# 5. Plots de depuración
# ============================================================

def plot_segmentation_debug(img_rgb, mask_final, debug, contour_info=None, figsize=(22, 14)):
    """
    Muestra plots intermedios para ajustar umbrales.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    final_overlay = overlay_mask(img_rgb, mask_final, color=(255, 255, 0), alpha=0.45)

    if contour_info is not None:
        contour_overlay = draw_contour_overlay(
            img_rgb,
            contour_info["contour"],
            color=(255, 255, 0),
            thickness=2,
            show_vertices=True,
        )
    else:
        contour_overlay = img_rgb.copy()

    fig, axes = plt.subplots(3, 4, figsize=figsize)
    axes = axes.ravel()

    axes[0].imshow(img_rgb)
    axes[0].set_title("Original level usado")
    axes[0].axis("off")

    axes[1].imshow(debug["hsv_S"], cmap="gray")
    axes[1].set_title("Saturación HSV")
    axes[1].axis("off")

    axes[2].imshow(debug["hsv_V"], cmap="gray")
    axes[2].set_title("Value HSV")
    axes[2].axis("off")

    im3 = axes[3].imshow(debug["dist_to_bg"], cmap="magma")
    axes[3].set_title("Distancia RGB al fondo")
    axes[3].axis("off")
    plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    im4 = axes[4].imshow(debug["od_mean"], cmap="magma")
    axes[4].set_title("Optical density media")
    axes[4].axis("off")
    plt.colorbar(im4, ax=axes[4], fraction=0.046, pad=0.04)

    axes[5].imshow(debug["black_padding_mask"], cmap="gray")
    axes[5].set_title("Padding negro conectado a borde")
    axes[5].axis("off")

    axes[6].imshow(debug["mask_sat"], cmap="gray")
    axes[6].set_title("Candidato por saturación")
    axes[6].axis("off")

    axes[7].imshow(debug["mask_dist"], cmap="gray")
    axes[7].set_title("Candidato por distancia al fondo")
    axes[7].axis("off")

    axes[8].imshow(debug["mask_od"], cmap="gray")
    axes[8].set_title("Candidato por OD")
    axes[8].axis("off")

    axes[9].imshow(debug["raw_candidate_mask"], cmap="gray")
    axes[9].set_title("Candidato combinado")
    axes[9].axis("off")

    axes[10].imshow(final_overlay)
    axes[10].set_title("Máscara final sobre original")
    axes[10].axis("off")

    axes[11].imshow(contour_overlay)
    axes[11].set_title("Contorno global cerrado")
    axes[11].axis("off")

    plt.tight_layout()
    plt.show()


def plot_final_contour_large(img_rgb, contour_info, figsize=(10, 14)):
    """Plot grande del contorno final."""
    overlay = draw_contour_overlay(
        img_rgb,
        contour_info["contour"],
        color=(255, 255, 0),
        thickness=3,
        show_vertices=False,
    )

    plt.figure(figsize=figsize)
    plt.imshow(overlay)
    plt.title("Contorno global final del espécimen")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def plot_mask_and_contour(mask_final, contour_info, figsize=(10, 10)):
    """Muestra la máscara binaria y el contorno encima."""
    canvas = np.dstack([
        mask_final.astype(np.uint8) * 255,
        mask_final.astype(np.uint8) * 255,
        mask_final.astype(np.uint8) * 255,
    ])
    overlay = draw_contour_overlay(
        canvas,
        contour_info["contour"],
        color=(255, 0, 0),
        thickness=2,
    )

    plt.figure(figsize=figsize)
    plt.imshow(overlay)
    plt.title("Máscara final + contorno externo")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# ============================================================
# 6. Guardado de resultados
# ============================================================

def save_global_contour_outputs(result, output_dir):
    """
    Guarda máscara, overlays y puntos del contorno en level usado y level 0.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    img_rgb = result["img_level_rgb"]
    mask = result["mask_final"]
    contour = result["contour"]
    contour_level_xy = result["contour_level_xy"]
    contour_level0_xy = result["contour_level0_xy"]
    level = result["level"]

    mask_path = output_dir / f"mask_specimen_level{level}.png"
    contour_mask_path = output_dir / f"contour_mask_level{level}.png"
    overlay_path = output_dir / f"overlay_contour_level{level}.png"
    contour_csv_path = output_dir / f"contour_points_level{level}.csv"
    contour_level0_csv_path = output_dir / "contour_points_level0.csv"
    metadata_path = output_dir / "metadata_global_contour.json"

    cv2.imwrite(str(mask_path), (mask.astype(np.uint8) * 255))
    cv2.imwrite(str(contour_mask_path), (result["contour_mask"].astype(np.uint8) * 255))

    overlay = draw_contour_overlay(img_rgb, contour, color=(255, 255, 0), thickness=2)
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(overlay_path), overlay_bgr)

    np.savetxt(
        contour_csv_path,
        contour_level_xy,
        delimiter=",",
        header="x_level,y_level",
        comments="",
        fmt="%.6f",
    )

    np.savetxt(
        contour_level0_csv_path,
        contour_level0_xy,
        delimiter=",",
        header="x_level0,y_level0",
        comments="",
        fmt="%.6f",
    )

    metadata = {
        "slide_path": str(result["slide_path"]),
        "level": int(result["level"]),
        "downsample": float(result["downsample"]),
        "level_shape_hw": [int(mask.shape[0]), int(mask.shape[1])],
        "area_px2_level": float(result["area_px2"]),
        "perimeter_px_level": float(result["perimeter_px"]),
        "bbox_level": result["bbox"],
        "centroid_level": result["centroid"],
        "num_contour_points_level": int(len(contour_level_xy)),
        "files": {
            "mask": str(mask_path),
            "contour_mask": str(contour_mask_path),
            "overlay": str(overlay_path),
            "contour_points_level": str(contour_csv_path),
            "contour_points_level0": str(contour_level0_csv_path),
        },
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print("\n==============================")
    print("GUARDADO")
    print("==============================")
    print(f"Carpeta: {output_dir}")
    print(f"Máscara: {mask_path}")
    print(f"Contorno máscara: {contour_mask_path}")
    print(f"Overlay: {overlay_path}")
    print(f"Puntos contorno level {level}: {contour_csv_path}")
    print(f"Puntos contorno level 0: {contour_level0_csv_path}")
    print(f"Metadata: {metadata_path}")

    return metadata


# ============================================================
# 7. Función principal
# ============================================================

def extract_he_global_specimen_contour(
    slide_path,
    target_level=None,
    max_analysis_side=3500,

    # Segmentación
    border_width_for_bg=10,
    dist_to_bg_threshold=30,
    sat_threshold=18,
    value_upper_threshold=245,
    od_threshold=0.10,
    min_saturation_for_od=5,
    black_v_threshold=35,
    black_mean_threshold=45,
    black_border_width=5,
    open_kernel_size=5,
    close_kernel_size=21,
    smooth_kernel_size=9,
    keep_n_components=1,
    min_component_area_px=5000,
    min_component_area_frac=0.0005,
    fill_internal_holes=True,

    # Contorno
    approx_epsilon_frac=0.001,
    contour_line_thickness=2,

    # Visualización / salida
    show_debug_plots=True,
    show_final_large=True,
    save_outputs=False,
    output_dir=None,
):
    """
    Pipeline completo:
    1. Lee la WSI en un level bajo/medio.
    2. Segmenta el espécimen completo.
    3. Extrae un contorno externo global cerrado.
    4. Devuelve coordenadas del contorno en el level usado y en level 0.
    """
    slide_path = Path(slide_path)

    slide, img_rgb, level, downsample = read_wsi_level(
        slide_path,
        target_level=target_level,
        max_analysis_side=max_analysis_side,
    )

    mask_final, debug = segment_he_tissue_global(
        img_rgb,
        border_width_for_bg=border_width_for_bg,
        dist_to_bg_threshold=dist_to_bg_threshold,
        sat_threshold=sat_threshold,
        value_upper_threshold=value_upper_threshold,
        od_threshold=od_threshold,
        min_saturation_for_od=min_saturation_for_od,
        black_v_threshold=black_v_threshold,
        black_mean_threshold=black_mean_threshold,
        black_border_width=black_border_width,
        open_kernel_size=open_kernel_size,
        close_kernel_size=close_kernel_size,
        smooth_kernel_size=smooth_kernel_size,
        keep_n_components=keep_n_components,
        min_component_area_px=min_component_area_px,
        min_component_area_frac=min_component_area_frac,
        fill_internal_holes=fill_internal_holes,
    )

    contour_info = extract_external_contour_from_mask(
        mask_final,
        approx_epsilon_frac=approx_epsilon_frac,
        contour_line_thickness=contour_line_thickness,
    )

    contour_level_xy = contour_to_xy(contour_info["contour"])
    contour_level0_xy = scale_contour_to_level0(contour_info["contour"], downsample=downsample)

    result = {
        "slide": slide,
        "slide_path": slide_path,
        "img_level_rgb": img_rgb,
        "level": level,
        "downsample": downsample,
        "mask_final": mask_final,
        "debug": debug,
        "raw_contour": contour_info["raw_contour"],
        "contour": contour_info["contour"],
        "contour_mask": contour_info["contour_mask"],
        "contour_level_xy": contour_level_xy,
        "contour_level0_xy": contour_level0_xy,
        "area_px2": contour_info["area_px2"],
        "perimeter_px": contour_info["perimeter_px"],
        "bbox": contour_info["bbox"],
        "centroid": contour_info["centroid"],
        "contour_info": contour_info,
    }

    print("\n==============================")
    print("RESULTADO LISTO PARA REGISTRO")
    print("==============================")
    print(f"contour_level_xy shape: {contour_level_xy.shape}")
    print(f"contour_level0_xy shape: {contour_level0_xy.shape}")
    print("Primeros 5 puntos en level usado:")
    print(contour_level_xy[:5])
    print("Primeros 5 puntos en level 0:")
    print(contour_level0_xy[:5])

    if show_debug_plots:
        plot_segmentation_debug(
            img_rgb,
            mask_final,
            debug,
            contour_info=contour_info,
            figsize=(22, 14),
        )
        plot_mask_and_contour(mask_final, contour_info, figsize=(10, 10))

    if show_final_large:
        plot_final_contour_large(img_rgb, contour_info, figsize=(10, 14))

    if save_outputs:
        if output_dir is None:
            output_dir = slide_path.parent / "global_contour_outputs"
        save_global_contour_outputs(result, output_dir=output_dir)

    return result


# ============================================================
# 8. Ejemplo de uso
# ============================================================

if __name__ == "__main__":
    slide_path = r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs"

    result = extract_he_global_specimen_contour(
        slide_path=slide_path,

        # Puedes fijar target_level=4 como antes.
        # Si lo dejas en None, el código elige un level manejable automáticamente.
        target_level=4,
        max_analysis_side=3500,

        # ---- Umbrales principales a tocar si falla ----
        dist_to_bg_threshold=30,
        sat_threshold=18,
        od_threshold=0.10,

        # ---- Morfología ----
        open_kernel_size=5,
        close_kernel_size=21,
        smooth_kernel_size=9,

        # Si el espécimen son varias piezas separadas, prueba keep_n_components=2, 3 o None.
        keep_n_components=1,

        # Simplificación del contorno.
        # Baja a 0.0003 si quieres más detalle; sube a 0.002 si quieres más suavizado.
        approx_epsilon_frac=0.001,

        show_debug_plots=True,
        show_final_large=True,

        save_outputs=False,
        output_dir=r"Datos\SB013\SB013\contorno_global_HE",
    )

    # Contorno en coordenadas del level usado:
    contour_level_xy = result["contour_level_xy"]

    # Contorno escalado a level 0:
    contour_level0_xy = result["contour_level0_xy"]

    # Máscara global del espécimen en el level usado:
    mask_specimen = result["mask_final"]
